# AlegroCode — Colab GPU Server

| Ячейка | Что делает | Когда запускать |
|--------|-----------|------------------|
| **1 — INSTALL** | Клонирует репо, задаёт env vars, ставит зависимости | Один раз после запуска среды |
| **2 — START** | Скачивает веса (параллельно), загружает модели, поднимает сервер + ngrok | При каждом запуске или **рестарте** |
| **3 — STOP** | Останавливает сервер (модели остаются в памяти) | Когда нужно остановить |

> **Рестарт без перезагрузки ядра**: повторно запустите ячейку **2 — START**.
> Модели не перезагружаются. Веса не скачиваются повторно (кэш HuggingFace).
>
> **Единственное что нужно заполнить**: блок `ЗАПОЛНИТЕ ЭТИ ЗНАЧЕНИЯ` в ячейке 1.


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  1 — INSTALL  (один раз на сессию)                             ║
# ╚══════════════════════════════════════════════════════════════════╝

# ┌─────────────────────────────────────────────────────────────────┐
# │  ЗАПОЛНИТЕ ЭТИ ЗНАЧЕНИЯ ПЕРЕД ЗАПУСКОМ                         │
# └─────────────────────────────────────────────────────────────────┘
REPO_URL         = 'https://github.com/YOUR_USER/building_analyzer.git'
NGROK_AUTHTOKEN  = ''        # токен с dashboard.ngrok.com/get-started/your-authtoken
INPAINT_PROVIDER = 'lama'   # 'lama' или 'sd'
DB_PATH          = '/work/building_analyzer/data/alegrocode.db'
SCRAPER_ENABLED  = 'false'  # 'true' / 'false'
# ──────────────────────────────────────────────────────────────────

import os, sys, subprocess, textwrap
from pathlib import Path

# ── 1. Клонируем репозиторий (если ещё не скачан) ────────────────────────
repo_root = Path('/work/building_analyzer')
if not repo_root.exists():
    print('Клонируем репозиторий...')
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(repo_root)], check=True)
    print('Репозиторий склонирован в', repo_root)
else:
    print('Репозиторий уже есть:', repo_root)

os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# ── 2. Выставляем переменные окружения ───────────────────────────────────
if NGROK_AUTHTOKEN:
    os.environ['ALEGRO_NGROK_AUTHTOKEN'] = NGROK_AUTHTOKEN
os.environ['ALEGRO_INPAINT_PROVIDER'] = INPAINT_PROVIDER
os.environ['ALEGRO_DB_PATH']           = DB_PATH
os.environ['ALEGRO_SCRAPER_ENABLED']   = SCRAPER_ENABLED
os.environ.pop('PIP_CONSTRAINT', None)
print('Env vars: OK')

# ── 3. Pip tools ─────────────────────────────────────────────────────────
%pip install -q "pip>=24,<26.1" setuptools wheel

# ── 4. Фиксируем packaging ДО любых других установок ─────────────────────
# nvidia-dali и Grounding DINO требуют packaging<=24.2
%pip install -q --no-deps "packaging<=24.2"

# ── 5. Constraints-файл ───────────────────────────────────────────────────
(repo_root / 'constraints-colab.txt').write_text(textwrap.dedent("""
    numpy>=1.26,<2.0
    opencv-python-headless==4.10.0.84
    matplotlib>=3.8,<3.11
    packaging<=24.2
    scipy>=1.12,<2.0
    transformers>=4.44,<4.46
    accelerate>=0.33,<0.35
    tokenizers>=0.19,<0.20
    huggingface-hub>=0.24,<1.0
    tqdm>=4.66,<5
""").strip() + '\n', encoding='utf-8')

# ── 6. Server-critical пакеты (устанавливаем первыми — независимо от ML) ─
%pip install -q --prefer-binary -c constraints-colab.txt \
    "fastapi==0.115.*" "uvicorn[standard]==0.34.*" \
    python-multipart "sqlalchemy>=2,<3" aiosqlite \
    "pydantic>=2.7" "pydantic-settings>=2.3" \
    "APScheduler>=3.10" httpx pyngrok nest_asyncio

# ── 7. HF/CV стек ────────────────────────────────────────────────────────
%pip install -q --prefer-binary -c constraints-colab.txt \
    "numpy>=1.26,<2.0" "opencv-python-headless==4.10.0.84" "matplotlib>=3.8,<3.11" \
    "transformers>=4.44,<4.46" "accelerate>=0.33,<0.35" \
    "tokenizers>=0.19,<0.20" "huggingface-hub>=0.24,<1.0"

# ── 8. scipy с --only-binary ─────────────────────────────────────────────
%pip install -q --only-binary :all: -c constraints-colab.txt "scipy>=1.12,<2.0"

# ── 9. scikit-image ───────────────────────────────────────────────────────
%pip install -q --prefer-binary "scikit-image>=0.22"

# ── 10. SAM2 runtime deps (отсутствие вызывало WARNING при запуске) ───────
%pip install -q --prefer-binary "hydra-core>=1.3.2" "iopath>=0.1.10"

# ── 11. curl_cffi — нужен scraper (curl TLS fingerprinting) ─────────────
%pip install -q --prefer-binary "curl_cffi>=0.7"

# ── 12. Остальные зависимости из requirements.txt ─────────────────────────
src_req = repo_root / 'backend' / 'requirements.txt'
flt_req = repo_root / 'backend' / 'requirements.colab.filtered.txt'
skip = (
    'torch', 'torchvision', 'transformers', 'accelerate', 'tokenizers',
    'huggingface-hub', 'numpy', 'opencv-python-headless', 'matplotlib',
    'scikit-image', 'scipy',
    'fastapi', 'uvicorn', 'python-multipart', 'sqlalchemy', 'aiosqlite',
    'pydantic', 'APScheduler', 'httpx', 'pyngrok', 'nest_asyncio',
    'hydra-core', 'iopath', 'curl_cffi',
)
lines = []
for line in src_req.read_text(encoding='utf-8').splitlines():
    s = line.strip()
    if not s or s.startswith('#') or not s.startswith(skip):
        lines.append(line)
flt_req.write_text('\n'.join(lines) + '\n', encoding='utf-8')
%pip install -q --prefer-binary -r backend/requirements.colab.filtered.txt -c constraints-colab.txt

# ── 13. SAM2 ──────────────────────────────────────────────────────────────
%pip install -q --no-build-isolation --no-deps git+https://github.com/facebookresearch/sam2.git

# ── 14. hf_transfer — ускоряет загрузку весов моделей с HuggingFace ~3-5x ─
%pip install -q hf_transfer

# ── 15. Переустанавливаем numpy последним — некоторые пакеты выше его бьют
%pip install -q --force-reinstall --prefer-binary "numpy>=1.26,<2.0"

import os; os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
import nest_asyncio; nest_asyncio.apply()
import numpy, cv2, sqlalchemy, fastapi
print(f"numpy {numpy.__version__} | cv2 {cv2.__version__} | "
      f"fastapi {fastapi.__version__} | sqlalchemy {sqlalchemy.__version__}")
print("nest_asyncio: OK | hf_transfer: ON")
print("\n✅  Установка завершена — запустите ячейку 2 (START)")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  2 — START  (запуск или рестарт сервера)                       ║
# ╚══════════════════════════════════════════════════════════════════╝
#
# Запускайте повторно:
#   • остановит старый сервер  • освободит порт 8000
#   • перезапустит ngrok       • НЕ перезагружает модели (_analyzer)

import asyncio, subprocess, os, time as _time
import nest_asyncio
nest_asyncio.apply()
os.environ.setdefault('HF_HUB_ENABLE_HF_TRANSFER', '1')

ngrok_token = os.environ.get('ALEGRO_NGROK_AUTHTOKEN', '')
if not ngrok_token:
    print('⚠️  ALEGRO_NGROK_AUTHTOKEN не задан — ngrok не подключится.')

# ── Останавливаем старый сервер ───────────────────────────────────────────
_server = globals().get('_server')
_server_task = globals().get('_server_task')
if _server is not None:
    print('Останавливаем предыдущий сервер...')
    _server.should_exit = True
    try:
        await asyncio.wait_for(_server_task, timeout=8)
    except Exception:
        pass
    print('Предыдущий сервер остановлен')

subprocess.run(
    "fuser -k 8000/tcp 2>/dev/null || lsof -ti:8000 | xargs kill -9 2>/dev/null || true",
    shell=True, capture_output=True
)
await asyncio.sleep(0.5)

# ── Отключить старый ngrok ────────────────────────────────────────────────
_tunnel = globals().get('_tunnel')
if _tunnel is not None:
    try:
        from pyngrok import ngrok as _ng
        _ng.disconnect(_tunnel.public_url)
        _ng.kill()
    except Exception:
        pass

# ── Скачиваем веса параллельно (только если не в кэше) ───────────────────
# snapshot_download проверяет кэш — повторный запуск мгновенный.
# Параллельная загрузка 3 моделей одновременно вместо последовательной.
_analyzer = globals().get('_analyzer')
if _analyzer is None:
    import concurrent.futures
    from huggingface_hub import snapshot_download
    _MODEL_IDS = [
        'IDEA-Research/grounding-dino-base',
        'facebook/sam-vit-base',
        'CIDAS/clipseg-rd64-refined',
    ]
    _IGNORE = ['*.msgpack', '*.h5', 'flax_model*', 'tf_model*', '*.ot', 'rust_model*']
    print('Скачиваем веса 3 моделей параллельно...')
    _t0 = _time.time()
    with concurrent.futures.ThreadPoolExecutor(max_workers=3) as _pool:
        _futs = {_pool.submit(snapshot_download, m, ignore_patterns=_IGNORE): m
                 for m in _MODEL_IDS}
        for _f in concurrent.futures.as_completed(_futs):
            _m = _futs[_f]
            try:
                _f.result()
                print(f'  ✓ {_m}')
            except Exception as _e:
                print(f'  ⚠ {_m}: {_e}')
    print(f'Веса готовы за {_time.time()-_t0:.1f}s')

# ── Загружаем модели (из локального кэша — быстро) ───────────────────────
if _analyzer is None:
    import sys
    from pathlib import Path
    repo_root = Path('/work/building_analyzer')
    if not repo_root.exists():
        repo_root = Path.cwd().resolve()
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))
    print('Загружаем модели в память GPU...')
    _t0 = _time.time()
    from backend.ml_pipeline import FacadeAnalyzer
    _analyzer = FacadeAnalyzer()
    _analyzer.load_models()
    print(f'Модели загружены на {_analyzer.device} за {_time.time()-_t0:.1f}s')
else:
    print(f'Модели уже в памяти (device={_analyzer.device}) — пропускаем загрузку')

# ── Создаём FastAPI приложение ────────────────────────────────────────────
# Сбрасываем кэш настроек чтобы подхватить актуальные env vars
from backend.core.config import get_settings
get_settings.cache_clear()
from backend.api import create_app
_app = create_app(analyzer=_analyzer)

# ── Запускаем сервер ─────────────────────────────────────────────────────
import uvicorn
_config = uvicorn.Config(_app, host='0.0.0.0', port=8000, loop='asyncio', log_level='info')
_server = uvicorn.Server(_config)
_server_task = asyncio.ensure_future(_server.serve())

# ── Health check через async httpx — НЕ блокирует event loop ─────────────
# ВАЖНО: синхронный httpx.get(...) блокирует Python thread, и uvicorn
# не может обработать запрос (оба в одном event loop). Используем async.
import httpx
await asyncio.sleep(1)  # первый тик — uvicorn биндит сокет
async with httpx.AsyncClient() as _hc:
    for _i in range(30):
        try:
            _r = await _hc.get('http://127.0.0.1:8000/api/health', timeout=3)
            if _r.status_code == 200:
                print(f'Сервер запущен (попытка {_i+1}):', _r.json())
                break
        except httpx.RequestError:
            pass
        await asyncio.sleep(1)
    else:
        raise RuntimeError('Сервер не поднялся за 30 сек — проверьте логи выше')

# ── Подключаем ngrok ──────────────────────────────────────────────────────
_tunnel = None
if ngrok_token:
    from pyngrok import ngrok
    ngrok.set_auth_token(ngrok_token)
    _tunnel = ngrok.connect(8000, 'http')
    print('=' * 60)
    print('PUBLIC URL:', _tunnel.public_url)
    print('DOCS      :', _tunnel.public_url + '/docs')
    print('HEALTH    :', _tunnel.public_url + '/api/health')
    print('=' * 60)
    print('Вставьте PUBLIC URL в настройках Flutter-приложения.')
else:
    print('Сервер доступен локально: http://127.0.0.1:8000')
    print('Ngrok не подключён (нет токена)')


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  3 — STOP  (остановить сервер, модели останутся в памяти)      ║
# ╚══════════════════════════════════════════════════════════════════╝
import asyncio

_server = globals().get('_server')
_server_task = globals().get('_server_task')
_tunnel = globals().get('_tunnel')

if _server is not None:
    _server.should_exit = True
    try:
        await asyncio.wait_for(_server_task, timeout=10)
    except asyncio.TimeoutError:
        _server_task.cancel()
    _server = None
    print('Сервер остановлен')
else:
    print('Сервер не запущен')

if _tunnel is not None:
    try:
        from pyngrok import ngrok
        ngrok.disconnect(_tunnel.public_url)
        ngrok.kill()
    except Exception as e:
        print('ngrok.disconnect:', e)
    _tunnel = None
    print('Ngrok отключён')

print('Модели остаются в памяти — можно снова запустить ячейку 2 (START)')


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  4 — SCRAPER  (опционально, разовый запуск парсера цен)        ║
# ╚══════════════════════════════════════════════════════════════════╝
from backend.scraper.worker import run_once
await run_once('all')
